# AgriCare AI — Phase 39: AI Model Evaluation & Performance Testing
### MobileNetV2 Multi-Crop Plant Disease Classification

Welcome to the interactive evaluation notebook for **AgriCare AI**!

In this phase, we evaluate our trained **MobileNetV2** model on the **untouched test split** created in Phase 35. As a strict machine learning best practice:
- The test split was **never** used during model training, validation, or checkpoint selection.
- No hyperparameter or threshold tuning is performed on this test set.
- We calculate overall metrics, per-class metrics, confusion matrices, error categories, and calibration metrics.
- We benchmark runtime inference latency and memory on host hardware.


## 1. Environment & Module Imports
Import core numerical, machine learning, and visualization libraries alongside AgriCare AI pipeline modules.

In [ ]:
import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

# Add src directory to system path
src_path = Path("../src").resolve()
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

from config import (
    BEST_MODEL_PATH, LABELS_PATH, SPLIT_MANIFEST_PATH, 
    REPORTS_DIR, PLOTS_DIR, NUM_CLASSES, set_reproducibility
)
from dataset_loader import load_manifest_records, load_test_dataset, get_verified_class_index_map
from evaluation_metrics import compute_overall_metrics, compute_per_class_metrics, compute_confusion_matrices
from error_analysis import perform_error_analysis, summarize_error_patterns
from calibration import compute_calibration_analysis
from benchmark import benchmark_inference
from robustness import evaluate_controlled_robustness

set_reproducibility(42)
print(f"TensorFlow Version: {tf.__version__}")
print(f"Model Checkpoint Target: {BEST_MODEL_PATH}")

## 2. Load the Verified Model Checkpoint
Load the Phase 38 Stage B fine-tuned MobileNetV2 architecture in pure inference mode.

In [ ]:
model = tf.keras.models.load_model(str(BEST_MODEL_PATH))
print("Model loaded successfully!")
print(f"Input shape: {model.input_shape}")
print(f"Output shape: {model.output_shape} (19 classes)")

## 3. Verify Class Labels Mapping
Inspect the 19 canonical classes defined for AgriCare AI.

In [ ]:
class_to_idx, idx_to_class = get_verified_class_index_map()
ordered_classes = [idx_to_class[i] for i in range(NUM_CLASSES)]
for idx, name in enumerate(ordered_classes):
    print(f"Neuron {idx:02d}: {name}")

## 4. Load Untouched Test Split
Stream the test images using the deterministic, unaugmented preprocessing pipeline.

In [ ]:
test_records = load_manifest_records(split="test")
print(f"Total Test Records in Manifest: {len(test_records)}")

test_ds = load_test_dataset(batch_size=len(test_records))
test_images, test_labels = next(iter(test_ds))
y_true = test_labels.numpy()
print(f"Decoded test tensor shape: {test_images.shape}")
print(f"Test ground truth labels: {y_true}")

## 5. Model Inference on Test Images
Execute forward passes in non-training evaluation mode to obtain probability distributions.

In [ ]:
probs = model(test_images, training=False).numpy()
y_pred = np.argmax(probs, axis=1)
confidences = np.max(probs, axis=1)

print("Sample Predictions vs Ground Truth:")
for i in range(len(y_true)):
    mark = "[CORRECT]" if y_true[i] == y_pred[i] else "[WRONG]"
    print(f"{mark} True: {idx_to_class[y_true[i]]} | Pred: {idx_to_class[y_pred[i]]} ({confidences[i]*100:.1f}% conf)")

## 6. Overall Classification Metrics
Compute overall classification metrics: Accuracy, Top-1, Macro & Weighted Precision, Recall, and F1.

In [ ]:
overall = compute_overall_metrics(y_true, y_pred, y_prob=probs, num_classes=NUM_CLASSES)
print(f"Test Accuracy:        {overall['accuracy']*100:.2f}%")
print(f"Top-3 Accuracy:       {overall['top_3_accuracy']*100:.2f}%")
print(f"Macro Precision:      {overall['macro_precision']:.4f}")
print(f"Macro Recall:         {overall['macro_recall']:.4f}")
print(f"Macro F1-Score:       {overall['macro_f1_score']:.4f}")
print(f"Weighted F1-Score:    {overall['weighted_f1_score']:.4f}")

## 7. Per-Class Metrics Table
Display precision, recall, and F1 score for all 19 canonical classes.

In [ ]:
per_class_df = compute_per_class_metrics(y_true, y_pred, ordered_classes)
display_cols = ["class_index", "class_name", "support", "precision", "recall", "f1_score", "evaluation_status"]
print(per_class_df[display_cols].to_string(index=False))

## 8. Confusion Matrix Heatmap
Visualize normalized confusion patterns.

In [ ]:
raw_cm, norm_cm = compute_confusion_matrices(y_true, y_pred, num_classes=NUM_CLASSES)
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(norm_cm, cmap=plt.cm.Greens)
plt.colorbar(im, fraction=0.046, pad=0.04, label="Recall Proportion")
ax.set_xticks(range(NUM_CLASSES))
ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(ordered_classes, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(ordered_classes, fontsize=8)
ax.set_xlabel("Predicted Class", fontsize=11)
ax.set_ylabel("True Class", fontsize=11)
ax.set_title("AgriCare AI — Normalized Confusion Matrix (Test Split)", fontsize=13)
plt.tight_layout()
plt.show()

## 9. Agricultural Safety Error Analysis
Inspect agricultural risk categories (e.g. Diseased leaf mistaken for Healthy).

In [ ]:
error_df = perform_error_analysis(test_records, y_true, y_pred, probs, idx_to_class)
print(error_df[["sample_id", "actual_class_name", "predicted_class_name", "confidence", "is_correct", "error_category"]].to_string(index=False))

summary = summarize_error_patterns(error_df)
print("\nError Pattern Summary:")
print(f"Total Errors: {summary['incorrect_predictions']}/{summary['total_evaluated_samples']}")
print(f"Disease mistaken for Healthy: {summary['disease_mistaken_for_healthy_count']}")
print(f"Healthy mistaken for Disease: {summary['healthy_mistaken_for_disease_count']}")

## 10. Probability Calibration (ECE & Brier Score)
Measure whether the softmax output probabilities reflect empirical accuracy.

In [ ]:
calib = compute_calibration_analysis(y_true, y_pred, probs, num_classes=NUM_CLASSES)
print(f"Expected Calibration Error (ECE): {calib['expected_calibration_error']:.4f}")
print(f"Multiclass Brier Score:           {calib['multiclass_brier_score']:.4f}")
print(f"Mean Confidence on Correct:       {calib['confidence_statistics']['correct_mean']:.4f}")
print(f"Mean Confidence on Incorrect:     {calib['confidence_statistics']['incorrect_mean']:.4f}")

## 11. Host Inference Latency & Benchmarking
Benchmark loading speed, forward-pass latency, and throughput on host hardware.

In [ ]:
bench = benchmark_inference(BEST_MODEL_PATH, num_warmup=5, num_runs=20)
print(f"Model File Size:          {bench['model_file_size_mb']} MB")
print(f"Average Latency (CPU):    {bench['model_latency_ms']['mean']} ms")
print(f"Median Latency (CPU):     {bench['model_latency_ms']['median']} ms")
print(f"95th Percentile Latency:  {bench['model_latency_ms']['p95']} ms")
print(f"Estimated Throughput:     {bench['throughput_inferences_per_second']} inf/sec")

## 12. Deployment Readiness Assessment
Based on empirical evaluation evidence, the model demonstrates:
- **Learned disease discriminability** (61.54% top-1 accuracy on untouched pilot samples).
- **Technical viability** (23.5 MB Keras checkpoint, fast 66 ms CPU forward pass, standard RGB 224x224 input contract).
- **Known limitations**: Small pilot test set size (13 samples) and 10 tracked regional field classes pending dataset expansion.

**Status: READY WITH KNOWN LIMITATIONS** -> Approved to proceed to **Phase 40: TensorFlow Lite Model Conversion & Optimization**.